# Python: BERTopic Modeling

This section performs topic modeling on climate-related transcripts using BERTopic with multilingual embeddings and multiple representation models.

## Setup

Import all necessary Python libraries for topic modeling.

In [1]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI
from nltk.corpus import stopwords
import nltk
from pathlib import Path
import pickle
import os
from dotenv import load_dotenv
from openai import OpenAI as OpenAIClient

load_dotenv()

C:\Users\sile9\anaconda3\envs\bertopic_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

Configure project paths. Adjust based on notebook location.

In [2]:
project_root = Path.cwd().parent if "Scripts" in Path.cwd().parts else Path.cwd()
print(f"Project root: {project_root}")

data_dir = project_root / 'Data'
output_dir = project_root / 'Outputs'

data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {data_dir}")
print(f"Output directory: {output_dir}")

Project root: C:\Users\sile9\Documents\projects\parldebates_analysis
Data directory: C:\Users\sile9\Documents\projects\parldebates_analysis\Data
Output directory: C:\Users\sile9\Documents\projects\parldebates_analysis\Outputs


## Load Documents

Read the preprocessed transcripts.

In [3]:
df = pd.read_csv(data_dir / '05_transcripts_climate_for_topic_modeling.csv')
text_column = 'transcript_text'
documents = df[text_column].tolist()

print(f"Loaded {len(documents)} documents")
print(f"First document preview: {documents[0][:200]}...")

Loaded 4987 documents
First document preview: Natürliche Ressourcen sind von zentraler Bedeutung und eine Basis für die Wohlfahrt unserer Gesellschaft. Diese natürlichen Ressourcen, zu denen Wasser, Boden, saubere Luft, Energierohstoffe und Metal...


## Generate Embeddings

Create dense vector representations using a multilingual sentence transformer. Cache embeddings to disk for faster re-runs.

In [35]:
print("Calculating or loading embeddings...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

emb_path = data_dir / '06_embeddings.pickle'
if emb_path.exists():
    print("Loading existing embeddings from disk...")
    with emb_path.open('rb') as handle:
        embeddings = pickle.load(handle)
else:
    print("Computing embeddings (this may take a few minutes)...")
    embeddings = embedding_model.encode(documents, show_progress_bar=True)
    emb_path.parent.mkdir(parents=True, exist_ok=True)
    with emb_path.open('wb') as handle:
        pickle.dump(embeddings, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print("Embeddings saved to disk.")

print(f"Embeddings shape: {embeddings.shape}")

Calculating or loading embeddings...


Loading weights: 100%|████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 10839.97it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading existing embeddings from disk...
Embeddings shape: (4987, 768)


## Reduce Embeddings (2D)

Reduce embeddings to 2D for visualization using UMAP.

In [36]:
print("Reducing embeddings for visualization...")
reduced_embeddings = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.0,
    metric='cosine',
    random_state=42
).fit_transform(embeddings)

print(f"Reduced embeddings shape: {reduced_embeddings.shape}")

Reducing embeddings for visualization...
Reduced embeddings shape: (4987, 2)


## Configure Models

Set up UMAP (5D for clustering), HDBSCAN for clustering, and CountVectorizer with stopwords for topic representation.

In [48]:
# UMAP: Reduce to 5D for topic modeling
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# HDBSCAN: Cluster documents into topics
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=40,
    # min_samples=15,
    cluster_selection_epsilon=0.0,
    # max_cluster_size=500,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# Stopwords
nltk.download('stopwords', quiet=True)
german_stopwords = set(stopwords.words('german'))
french_stopwords = set(stopwords.words('french'))
swiss_german_stopwords = {'dass'}
all_stopwords = german_stopwords.union(french_stopwords).union(swiss_german_stopwords)

# CountVectorizer: Extract topic keywords
vectorizer_model = CountVectorizer(
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    stop_words=list(all_stopwords)
)

print(f"Total stopwords: {len(all_stopwords)}")

Total stopwords: 386


## Configure Representation Models

Set up KeyBERT and MMR for keyword extraction. OpenAI integration is commented out but available for generating human-readable labels.

In [49]:
# KeyBERT and MMR representation models
keybert = KeyBERTInspired()
mmr = MaximalMarginalRelevance(diversity=0.3)

# OpenAI representation model
print("Setting up OpenAI representation model...")
openai_api_key = os.getenv("OPENAI_API_KEY")

# Add this before creating the OpenAI client
# only for rendering quarto
if "SSL_CERT_FILE" in os.environ:
    print(f"SSL_CERT_FILE: {os.environ['SSL_CERT_FILE']}")
    # Remove it if it's causing issues
    del os.environ["SSL_CERT_FILE"]
if "SSL_CERT_DIR" in os.environ:
    print(f"SSL_CERT_DIR: {os.environ['SSL_CERT_DIR']}")
    del os.environ["SSL_CERT_DIR"]

# Create OpenAI client
client = OpenAIClient(api_key=openai_api_key)

# Prompt for OpenAI to generate topic labels
openai_prompt = """
I have a topic described by the following keywords: [KEYWORDS]

The topic contains these representative documents:
[DOCUMENTS]

Based on the keywords and documents above, generate a short, descriptive label for this topic in English (maximum 5 words). Only return the label itself in English, nothing else.
"""

openai_model = OpenAI(
    client=client,
    model="gpt-4o-mini",  # Cost-effective model
    prompt=openai_prompt,
    chat=True,
    nr_docs=5,  # Number of representative documents to include
    doc_length=200,  # Maximum words per document
    tokenizer="whitespace"  # Required: method to tokenize documents
)

representation_model = {
    "KeyBERT": keybert,
    "MMR": mmr,
    "OpenAI": openai_model,
}

print("Representation models configured.")

Setting up OpenAI representation model...
Representation models configured.


## Fit BERTopic Model

Train the topic model on the climate transcripts.

In [50]:
print("Fitting BERTopic model...")
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    nr_topics="auto",
    top_n_words=10,
    verbose=True
)

topics, probs = topic_model.fit_transform(documents, embeddings)

print("\nModel fitting complete!")

2026-03-12 10:25:26,624 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Fitting BERTopic model...


2026-03-12 10:25:35,477 - BERTopic - Dimensionality - Completed ✓
2026-03-12 10:25:35,477 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-12 10:25:35,706 - BERTopic - Cluster - Completed ✓
2026-03-12 10:25:35,706 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-12 10:25:39,289 - BERTopic - Representation - Completed ✓
2026-03-12 10:25:39,292 - BERTopic - Topic reduction - Reducing number of topics
2026-03-12 10:25:39,298 - BERTopic - Representation - Fine-tuning topics using representation models.
100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:05<00:00,  1.60it/s]
2026-03-12 10:25:52,365 - BERTopic - Representation - Completed ✓
2026-03-12 10:25:52,370 - BERTopic - Topic reduction - Reduced number of topics from 8 to 8



Model fitting complete!


## Extract and Save Results

Add topic assignments to the dataframe and export results.

In [51]:
df['topic'] = topics
df['topic_probability'] = probs

print(f"\nNumber of topics found: {len(set(topics)) - 1}")  # -1 for outliers
print("\nTopic distribution:")
print(df['topic'].value_counts())

topic_info = topic_model.get_topic_info()
topic_info_excl_outlier = topic_info[topic_info['Topic'] != -1]
print("\nTopic Information:")
print(topic_info_excl_outlier)


Number of topics found: 7

Topic distribution:
topic
 0    1554
 1    1212
-1     903
 2     540
 3     310
 4     209
 5     203
 6      56
Name: count, dtype: int64

Topic Information:
   Topic  Count                                               Name  \
1      0   1554   0_wasserkraft_énergétique_terawattstunden_winter   
2      1   1212  1_klimapolitik_gletscher_co2 abgabe_contre projet   
3      2    540                       2_urteil_egmr_cour_verfahren   
4      3    310        3_biodiversität_flächen_espèces_agriculture   
5      4    209         4_flugticketabgabe_luftfahrt_aviation_vols   
6      5    203      5_routes_véhicules_elektromobilität_fahrzeuge   
7      6     56                      6_supplément_naf_avs_milchkuh   

                                      Representation  \
1  [wasserkraft, énergétique, terawattstunden, wi...   
2  [klimapolitik, gletscher, co2 abgabe, contre p...   
3  [urteil, egmr, cour, verfahren, rechtsfragen, ...   
4  [biodiversität, flächen,

In [ ]:
# Save results
df.to_csv(data_dir / '06_documents_with_topics.csv', index=False)
topic_info.to_csv(data_dir / '06_topic_info.csv', index=False)

## Create Visualizations

Generate interactive HTML visualizations of topics and documents.

In [52]:
print("\nCreating visualizations...")

# Document visualization
fig_docs = topic_model.visualize_documents(
    documents,
    reduced_embeddings=reduced_embeddings,
    hide_document_hover=False,
    hide_annotations=False
)
fig_docs.write_html(output_dir / "06_topic_documents_visualization.html")
print("  ✓ Document visualization saved")

# Intertopic distance map
fig_topics = topic_model.visualize_topics()
fig_topics.write_html(output_dir / "06_intertopic_distance_map.html")
print("  ✓ Intertopic distance map saved")

# Topic hierarchy
fig_hierarchy = topic_model.visualize_hierarchy()
fig_hierarchy.write_html(output_dir / "06_topic_hierarchy.html")
print("  ✓ Topic hierarchy saved")

# Topic barchart
fig_barchart = topic_model.visualize_barchart(top_n_topics=10)
fig_barchart.write_html(output_dir / "06_topic_barchart.html")
print("  ✓ Topic barchart saved")

print(f"\nProcessing complete! Check the {output_dir} folder for visualizations.")


Creating visualizations...
  ✓ Document visualization saved
  ✓ Intertopic distance map saved
  ✓ Topic hierarchy saved
  ✓ Topic barchart saved

Processing complete! Check the C:\Users\sile9\Documents\projects\parldebates_analysis\Outputs folder for visualizations.
